In [2]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")


In [3]:
from langchain.tools import tool
@tool
def get_weather(location:str)->str:
    """Get the weather of a location"""
    return f"its sunny in {location}"
model_with_tool=model.bind_tools([get_weather])

In [6]:
response=model_with_tool.invoke("what is the weather in delhi")
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"args: {tool_call['args']}")
    print(response)

Tool: get_weather
args: {'location': 'Delhi'}
content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Delhi. I need to use the get_weather function. The function requires a location parameter. Delhi is the location here. So I should call get_weather with location set to "Delhi". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'pa1m95bye', 'function': {'arguments': '{"location":"Delhi"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 153, 'total_tokens': 249, 'completion_time': 0.150589386, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.009877921, 'prompt_tokens_details': None, 'queue_time': 0.987980036, 'total_time': 0.160467307}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs

In [9]:
#tool execution loop
#1 model generate tool calls
messages=[{"role":"user","content":"whats the weather in boston"}]
ai_msg=model_with_tool.invoke(messages)
messages.append(ai_msg)
#2 execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)
#3 pass result back to the final message
final_response=model_with_tool.invoke(messages)
print(final_response.text)

The weather in Boston is currently sunny. Let me know if you need more details! ☀️
